In [1]:
pip install datasets pandas

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 15.4 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 15.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 21.2 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 10.8 MB/s  0:00:03m0:00:0100:01
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  9/33 [idna]ow]
    Found exist

In [5]:
pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
from datasets import load_dataset
import pandas as pd
import random
import json
import re
import os
from collections import defaultdict

# =========================================================
# LOAD OPENR1 DATASET (STREAMING)
# =========================================================

dataset = load_dataset("open-r1/OpenR1-Math-220k",split="train",streaming=True)

# =========================================================
# TARGET SAMPLE SIZE
# =========================================================

TOTAL_SAMPLES = 7500

# =========================================================
# TARGET problem_type DISTRIBUTION
# =========================================================

problem_distribution = {
    "Algebra": 0.50888,
    "Other": 0.18472,
    "Calculus": 0.11279,
    "Geometry": 0.10657,
    "Number Theory": 0.05329,
    "Combinatorics": 0.02487,
    "Inequalities": 0.00622,
    "Logic and Puzzle": 0.00266
}

# =========================================================
# TARGET question_type DISTRIBUTION
# 0 = FRQ
# 1 = MCQ
# =========================================================

question_distribution = {
    0: 0.666963,
    1: 0.333037
}

# =========================================================
# CALCULATE NESTED TARGETS
# =========================================================

nested_targets = {}

for ptype, pprob in problem_distribution.items():

    total_for_type = int(pprob * TOTAL_SAMPLES)

    nested_targets[ptype] = {
        0: int(total_for_type * question_distribution[0]),
        1: int(total_for_type * question_distribution[1])
    }

print("\nNested Targets:")
print(json.dumps(nested_targets, indent=2))

# =========================================================
# SYSTEM PROMPT
# =========================================================

SYSTEM_PROMPT = (
    "You are a mathematical reasoning assistant. "
    "Solve the problem step-by-step using clear reasoning "
    "inside <think>...</think> tags and provide the final "
    "answer inside \\boxed{}."
)

# =========================================================
# MCQ DETECTOR
# MCQ options appear INSIDE problem text
# =========================================================

def is_mcq(problem_text):

    if not isinstance(problem_text, str):
        return False

    patterns = [
        r"\(A\)",
        r"\(B\)",
        r"\(C\)",
        r"\(D\)",
        r"A\.",
        r"B\.",
        r"C\.",
        r"D\."
    ]

    matches = sum(bool(re.search(p, problem_text)) for p in patterns)

    return matches >= 3

# =========================================================
# STORAGE
# =========================================================

sampled_rows = []

# metadata only for verification
metadata_rows = []

# counts for nested stratified sampling
counts = defaultdict(lambda: defaultdict(int))

# =========================================================
# STREAM + STRATIFIED SAMPLE
# =========================================================

for row in dataset:

    # -----------------------------------------------------
    # GET problem_type
    # -----------------------------------------------------

    ptype = row.get("problem_type", "Other")

    if ptype not in nested_targets:
        continue

    # -----------------------------------------------------
    # GET QUESTION TEXT
    # -----------------------------------------------------

    problem_text = row.get("problem", "")

    # -----------------------------------------------------
    # DETERMINE MCQ / FRQ
    # -----------------------------------------------------

    qtype = 1 if is_mcq(problem_text) else 0

    # -----------------------------------------------------
    # CHECK NESTED QUOTAS
    # -----------------------------------------------------

    if counts[ptype][qtype] >= nested_targets[ptype][qtype]:
        continue

    # -----------------------------------------------------
    # GET REASONING TRACE
    # -----------------------------------------------------

    reasoning = ""

    generations = row.get("generations", [])

    if isinstance(generations, list) and len(generations) > 0:
        reasoning = generations[0]

    # -----------------------------------------------------
    # GET FINAL ANSWER
    # -----------------------------------------------------

    answer = row.get("answer", "")

    assistant_output = (
        f"{reasoning}\n\n"
        f"\\boxed{{{answer}}}"
    )

    # -----------------------------------------------------
    # FINAL FORMAT
    # ONLY 2 COLUMNS
    # -----------------------------------------------------

    formatted_row = {

        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": problem_text
            }
        ],

        "completion": [
            {
                "role": "assistant",
                "content": assistant_output
            }
        ]
    }

    sampled_rows.append(formatted_row)

    # -----------------------------------------------------
    # SAVE METADATA FOR VERIFICATION
    # -----------------------------------------------------

    metadata_rows.append({
        "problem_type": ptype,
        "question_type": qtype
    })

    # -----------------------------------------------------
    # UPDATE COUNTS
    # -----------------------------------------------------

    counts[ptype][qtype] += 1

    # -----------------------------------------------------
    # STOP CONDITION
    # -----------------------------------------------------

    if len(sampled_rows) >= TOTAL_SAMPLES:
        break

# =========================================================
# SHUFFLE
# =========================================================

random.shuffle(sampled_rows)

# =========================================================
# SAVE JSONL
# =========================================================

output_path = os.path.expanduser(
    "~/Downloads/openr1_math_7_5k_stratified.jsonl"
)

with open(output_path, "w") as f:

    for row in sampled_rows:
        f.write(json.dumps(row) + "\n")

print(f"\nSaved dataset to:\n{output_path}")

# =========================================================
# VERIFY DISTRIBUTIONS
# =========================================================

metadata_df = pd.DataFrame(metadata_rows)

print("\nProblem Type Distribution:")
print(
    metadata_df["problem_type"]
    .value_counts(normalize=True)
)

print("\nQuestion Type Distribution:")
print(
    metadata_df["question_type"]
    .value_counts(normalize=True)
)

print("\nTotal Samples:")
print(len(metadata_df))

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]


Nested Targets:
{
  "Algebra": {
    "0": 2545,
    "1": 1270
  },
  "Other": {
    "0": 923,
    "1": 461
  },
  "Calculus": {
    "0": 563,
    "1": 281
  },
  "Geometry": {
    "0": 532,
    "1": 266
  },
  "Number Theory": {
    "0": 266,
    "1": 132
  },
  "Combinatorics": {
    "0": 124,
    "1": 61
  },
  "Inequalities": {
    "0": 30,
    "1": 15
  },
  "Logic and Puzzle": {
    "0": 12,
    "1": 6
  }
}

Saved dataset to:
/Users/deepaldeleena/Downloads/openr1_math_7_5k_stratified.jsonl

Problem Type Distribution:
problem_type
Algebra          0.571621
Geometry         0.119568
Other            0.115972
Calculus         0.098741
Number Theory    0.059634
Combinatorics    0.027720
Inequalities     0.006743
Name: proportion, dtype: float64

Question Type Distribution:
question_type
0    0.703626
1    0.296374
Name: proportion, dtype: float64

Total Samples:
6674


In [37]:
new_new = pd.read_json('/Users/deepaldeleena/Downloads/openr1_math_7_5k_stratified.jsonl', lines=True)


In [45]:
pd.set_option('display.max_colwidth', None)
display(new_new.iloc[[8]])

prompt  \
8  [{'role': 'system', 'content': 'You are a mathematical reasoning assistant. Solve the problem step-by-step using clear reasoning inside <think>...</think> tags and provide the final answer inside \boxed{}.'}, {'role': 'user', 'content': '13. Given $\frac{1}{3} \leqslant a \leqslant 1$. If $f(x)=a x^{2}-2 x+1$ has a maximum value $M(a)$ and a minimum value $N(a)$ on $[1,3]$, let $g(a)=M(a)-N(a)$.
(1) Find the function expression for $g(a)$;
(2) Prove that $g(a) \geqslant \frac{1}{2}$ always holds.'}]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      